In [2]:
import os
from dotenv import load_dotenv

# This looks for the .env file and loads it into your environment
load_dotenv() 

# Now you can access them safely
gemini_key = os.getenv("GEMINI_API_KEY")
mongo_uri = os.getenv("MONGO_URI")

if not gemini_key or not mongo_uri:
    print("❌ Error: Keys not found! Check your .env file.")

In [3]:
from google import genai
from google.genai import types
import os

# 1. Initialize with explicit key to rule out environment issues
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# 2. Use the current stable model (text-embedding-004 is deprecated)
model_id = "gemini-embedding-001" 

def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    try:
        response = client.models.embed_content(
            model=model_id,
            contents=text,
            config=types.EmbedContentConfig(
                task_type=task_type
            )
        )
        # Access the first embedding result
        return response.embeddings[0].values
    except Exception as e:
        print(f"Error caught: {e}")
        return None

# Test call
vector = get_embedding("abdul kalam")
if vector:
    print(f"Success! Vector length: {len(vector)}")

Success! Vector length: 3072


In [4]:
import os
import glob
import pdfplumber
from rapidocr_onnxruntime import RapidOCR
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Configuration
engine = RapidOCR()
directory_path = "./clg_pdfs"
all_docs = []

# 2. Get list of all PDF files in the directory
pdf_files = glob.glob(os.path.join(directory_path, "*.pdf"))
print(f"📂 Found {len(pdf_files)} PDFs in {directory_path}...")

# 3. Loop through every file
for file_path in pdf_files:
    file_name = os.path.basename(file_path)
    print(f"\n📄 Processing: {file_name}")
    
    try:
        with pdfplumber.open(file_path) as pdf:
            for i, page in enumerate(pdf.pages):
                # Try normal text extraction
                text = page.extract_text()
                
                # If page is empty/scanned, force OCR
                if not text or len(text.strip()) < 10:
                    print(f"  ⚠️ Page {i+1} is scanned. Running OCR...")
                    img = page.to_image(resolution=300).original
                    result, _ = engine(img)
                    if result:
                        text = "\n".join([line[1] for line in result])
                
                if text:
                    # We store the file_name in metadata so the AI can cite its source!
                    all_docs.append(Document(
                        page_content=text, 
                        metadata={
                            "source": file_name, 
                            "page": i+1,
                            "path": file_path
                        }
                    ))
    except Exception as e:
        print(f"  ❌ Error reading {file_name}: {e}")

# 4. Final Chunking
if all_docs:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    documents = text_splitter.split_documents(all_docs)
    print(f"\n✅ Total Files Processed: {len(pdf_files)}")
    print(f"✅ Total Chunks Created: {len(documents)}")
else:
    print("❌ No text could be extracted from any files.")

📂 Found 1 PDFs in ./clg_pdfs...

📄 Processing: documents-required.pdf

✅ Total Files Processed: 1
✅ Total Chunks Created: 2


In [5]:
texts = [doc.page_content for doc in documents]

print(f"✅ Ready to embed {len(texts)} chunks.") 

✅ Ready to embed 2 chunks.


In [6]:
import time
import re

# 1. Configuration for Free Tier
batch_size = 30  # Smaller batches to stay under TPM (Tokens Per Minute) limits
all_embeddings = []

print(f"🔄 Starting embedding for {len(texts)} chunks...")

# 2. Loop with error-aware retries
i = 0
while i < len(texts):
    mini_batch = texts[i : i + batch_size]
    print(f"📡 Processing chunks {i} to {min(i + batch_size, len(texts))}...")
    
    try:
        response = client.models.embed_content(
            model=model_id,
            contents=mini_batch,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
        )
        all_embeddings.extend([emb.values for emb in response.embeddings])
        i += batch_size # Move to next batch only on success
        
        # ☕ Mandatory 12-second pause to stay under ~5 RPM
        if i < len(texts):
            print("☕ Cooling down for 12 seconds to respect Free Tier limits...")
            time.sleep(12)
            
    except Exception as e:
        error_msg = str(e)
        if "429" in error_msg:
            # Look for "Please retry in X.Xs" in the error message
            wait_match = re.search(r"retry in (\d+\.?\d*)s", error_msg)
            wait_time = float(wait_match.group(1)) + 2 if wait_match else 40
            
            print(f"⚠️ Rate limit hit! API says wait {wait_time}s. Sleeping...")
            time.sleep(wait_time)
        else:
            print(f"❌ Unexpected Error: {e}")
            break

# 3. Final Verification
if len(all_embeddings) == len(texts):
    docs_to_insert = [
        {
            "text": texts[j],
            "embedding": all_embeddings[j],
            "metadata": documents[j].metadata 
        } 
        for j in range(len(all_embeddings))
    ]
    print(f"\n✅ Success! All {len(docs_to_insert)} chunks embedded.")

🔄 Starting embedding for 2 chunks...
📡 Processing chunks 0 to 2...

✅ Success! All 2 chunks embedded.


In [7]:
docs_to_insert

[{'text': 'List of Documents Required at the time of Admission\n1. MIETAdmissionFormOriginal&TwosetPhotocopy\n2. ThreesetPhotocopyofOriginalCounselingAllotmentLetter\n3. UPSEERankLetterPhotocopy\n4. ThreesetPhotocopyofStudentsAadharCard(Compulsory)\n5. ThreesetPhotocopyofStudentsPanCard\n6. TwoattestedcopyofvalidGATE/GPAT/CAT/MATScoreCard\n7. CounselingFeeReceipts\na) OriginalConfirmationfee\nb) AllXeroxCopyoffeedepositedatMIET\n8. ThreeattestedphotocopiesofMarksSheetofClassX\n9. ThreeattestedphotocopiesofCertificateofClassX\n10. ThreeattestedphotocopiesofMarkSheetofClassXII\n11. ThreeattestedphotocopiesofCertificateofClassXII\n12. ThreeattestedphotocopiesofMarkSheetofDiploma/B.Sc(forlateralentry)\n13. ThreeattestedMarkSheetofGraduation(foradmissioninpostGraduatecourse)\n14. ThreeattestedMarkSheetofPostGraduation(forM.Tech/M.Pharmonly)\n15. OriginalCharacterCertificate\n16. OriginalTransferCertificate(onlyforU.P.BoardStudents)&TwosetPhotocopy',
  'embedding': [0.02040974,
   -0.0100281

In [8]:
from pymongo import MongoClient
import os

# 1. Your connection string
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)

# 2. Define your actual project database and collection
# (This creates them automatically upon the first insertion)
db = client["rag_pdfs"] 
collection = db["pdfs"]

# 3. Perform the insertion
if docs_to_insert:
    result = collection.insert_many(docs_to_insert)
    print(f"✅ Success! Created database 'RAG_db' and inserted {len(result.inserted_ids)} documents.")
else:
    print("❌ Error: docs_to_insert is empty.")

✅ Success! Created database 'RAG_db' and inserted 2 documents.


In [9]:
result

InsertManyResult([ObjectId('69aa4e233e2c0f9eb29f98a4'), ObjectId('69aa4e233e2c0f9eb29f98a5')], acknowledged=True)

In [10]:
from pymongo.operations import SearchIndexModel
import time

# 1. Define the Index Name
index_name = "vector_index"

# 2. Create the Search Index Model
# Ensure numDimensions is 3072 for gemini-embedding-001
search_index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "numDimensions": 3072, # Correct for gemini-embedding-001
                "path": "embedding",
                "similarity": "cosine"
            }
        ]
    },
    name=index_name,
    type="vectorSearch"
)

# 3. Create the index on your collection
print(f"Creating index '{index_name}'...")
collection.create_search_index(model=search_index_model)

Creating index 'vector_index'...


'vector_index'

In [11]:
print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True

while True:
   indices = list(collection.list_search_indexes(index_name))
   if len(indices) and predicate(indices[0]):
      break
   time.sleep(5)
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


In [12]:
from google import genai
from google.genai import types

def get_query_results(query):
    """Gets results from a vector search query using Gemini and MongoDB."""
    
    # 1. Generate the query embedding using the modern SDK
    # We use 'RETRIEVAL_QUERY' to optimize for a short user question
    client = genai.Client()
    model_id = "gemini-embedding-001"
    
    response = client.models.embed_content(
        model=model_id,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
    )
    query_embedding = response.embeddings[0].values

    # 2. Define the MongoDB Vector Search pipeline
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",      # Must match your Atlas index name
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 100,         # Optimized (10-20x the limit)
                "limit": 5
            }
        },
        {
            "$project": {
                "_id": 0,
                "text": 1,
                "score": {"$meta": "vectorSearchScore"} # Useful for debugging
            }
        }
    ]

    # 3. Execute and collect results
    results = collection.aggregate(pipeline)
    
    # Converting cursor to a list
    array_of_results = list(results)
    
    print(f"Found {len(array_of_results)} relevant chunks.")
    return array_of_results


In [13]:
get_query_results("btech first year fees and doc required")

Found 5 relevant chunks.


[{'text': 'MEERUT INSTITUTE OF ENGINEERING & TECHNOLOGY\n(Approved by AICTE, Govt. of India; PCI(Pharmacy Council of India) &\naffiliated with Dr.APJ Abdul Kalam Technical University, Lucknow\nand Ch. Charan Singh University, Meerut)\nMEERUT – 250005\nFEES STRUCTURE (SESSION 2025 – 2026)\nINSTITUTE Career Student Regn. Fees Book Bank First Year Annual Fees\nCOURSE FEES Development Activities (one time) (one time) Fees (2nd Year\nFees @ Fees # onwards)\nB.TECH. ₹ 1,02,013/- ₹ 7,500/- ₹ 7,500/- ₹ 1,000/- ₹ 5,000/- ₹ 1,23,013/- ₹ 1,17,013/-\nM.TECH. ₹ 65,000/- N/A N/A ₹ 1,000/- N/A ₹ 66,000/- ₹ 65,000/-\nB.PHARM. ₹ 1,02,013/- ₹ 7,500/- ₹ 7,500/- ₹ 1,000/- N/A ₹ 1,18,013/- ₹ 1,17,013/-\nM.PHARM ₹ 90,000/- N/A N/A ₹ 1,000/- N/A ₹ 91,000/- ₹ 90,000/-\nMBA* ₹ 1,02,013/- N/A ₹ 7,500/- ₹ 1,000/- N/A ₹ 1,10,513/- ₹ 1,09,513/-\nMCA ₹ 1,02,013/- N/A ₹ 7,500/- ₹ 1,000/- N/A ₹ 1,10,513/- ₹ 1,09,513/-\nB.Sc. Hons\n₹ 30,630/- N/A ₹ 5,000/- ₹ 1,000/- N/A ₹ 36,630/- ₹ 35,630/-\n(BT/MB)\nB.Sc.',
  'score

In [17]:
from google import genai

# 1. Initialize the Gemini Client
client = genai.Client()

# 2. Retrieve your context using the function we built
query = "btech first year fees and doc required"
context_docs = get_query_results(query)

# 3. Cleanly join the text from your MongoDB results
context_string = "\n\n".join([doc["text"] for doc in context_docs])

# 4. Construct the prompt
# Pro-tip: Using "System Instructions" or clear separators improves accuracy
prompt = f"""
Use the following pieces of context to answer the question at the end. 
If the answer isn't in the context, say you don't know.

Context:
{context_string}

Question: {query}
"""

# 5. Generate the response using Gemini 2.0 Flash
# This replaces the openai.chat.completions.create call
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print("-" * 30)
print(response.text)

Found 5 relevant chunks.
------------------------------
**B.TECH. First Year Fees (Session 2025 – 2026):**

The total first year fees for B.TECH. is **₹ 1,31,513/-**.
This includes:
*   First Year Annual Fees (Institute): ₹ 1,23,013/-
*   University Registration Fees (one time): ₹ 1,000/-
*   University Examination Fees (per year): ₹ 7,500/-

**Documents Required at the time of Admission:**

1.  MIET Admission Form Original & Two set Photocopy
2.  Three set Photocopy of Original Counseling Allotment Letter
3.  UPSEE Rank Letter Photocopy
4.  Three set Photocopy of Students Aadhar Card (Compulsory)
5.  Three set Photocopy of Students Pan Card
6.  Two attested copy of valid GATE/GPAT/CAT/MAT Score Card
7.  Counseling Fee Receipts:
    a) Original Confirmation fee
    b) All Xerox Copy of fee deposited at MIET
8.  Three attested photocopies of Marks Sheet of Class X
9.  Three attested photocopies of Certificate of Class X
10. Three attested photocopies of Mark Sheet of Class XII
11. Three